# Optimising a lattice with SIMBA

Once you can track a machine, the next question is usually *what settings give the beam I want?* —
minimum emittance out of the injector, a matched linac, a target bunch length after the compressor,
and so on. `SIMBA` ships an optimisation subpackage (`simba.Modules.optimisation`) that wraps a
tracking run as an objective function, so any external optimiser can drive the machine.

This notebook covers two entry points:

1. **`xopt_optimisation`** — a ready-made evaluator that runs a full track for a set of variables and
   returns beam parameters, designed to plug straight into [Xopt](https://github.com/xopt-org/Xopt).
2. **`nelder_mead`** — a lightweight, dependency-free simplex optimiser for quick, small problems or
   when you want to write the objective yourself.

Both reuse everything from the [S2E chaining](start_to_end_chain.ipynb) example; here we optimise the
JFEL injector/linac from [laura-lattices](https://github.com/astec-stfc/laura-lattices).

### Setup

```bash
git clone https://github.com/astec-stfc/laura-lattices.git
export LATTICE_LOCATION=$(pwd)/laura-lattices/JFEL
export SIMCODES=/path/to/simcodes/directory
pip install xopt   # only needed for part 1
```

In [ ]:
import os
import numpy as np

## Part 1 — `xopt_optimisation` with Xopt

`xopt_optimisation(settings, directory, settings_file, ...)` does one full evaluation:

* `settings` — a dict of `"element:param": value` variables to apply before tracking
  (it may also carry a `"code"` key to swap the tracking code, as in the chaining example);
* it loads `settings_file` (a `.def`), applies the variables with `modifyElement`, tracks
  `start_lattice → end_lattice`, then loads every dumped beam and returns a flat dict of
  `"<beam>:<param>"` → value.

That return shape is exactly what Xopt's `evaluate_function` expects, so wiring it up is a matter of
declaring the variables and objective in a `VOCS`. Here we minimise the normalised horizontal
emittance at the end of the linac by varying two matching quadrupoles.

In [ ]:
from functools import partial
from simba.Modules.optimisation.xopt import xopt_optimisation

LATTICE = os.environ["LATTICE_LOCATION"]
SIMCODES = os.environ["SIMCODES"]

# Bind the fixed arguments so `evaluate` only takes the variable dict that Xopt supplies.
evaluate = partial(
    xopt_optimisation,
    directory="./opt_run",
    settings_file="Lattices/jfel_combined.def",
    start_lattice="generator",
    end_lattice="Linac",
    # forwarded to the internal Framework(...):
    master_lattice=LATTICE,
    simcodes=SIMCODES,
    generator_defaults="jfel.yaml",
    verbose=False,
)

Define the optimisation problem with a `VOCS`: the two quad strengths are the variables (with
bounds), and the objective is to minimise `enx` at the final beam dump. The variable keys use the
`element:param` convention that `xopt_optimisation` unpacks; the objective key must match one of the
`"<beam>:<param>"` entries the evaluator returns.

In [ ]:
from xopt import VOCS, Xopt, Evaluator
from xopt.generators.bayesian import ExpectedImprovementGenerator

vocs = VOCS(
    variables={
        "JFEL-S02-MAG-QUAD-01:k1l": [-2.0, 2.0],
        "JFEL-S02-MAG-QUAD-02:k1l": [-2.0, 2.0],
    },
    # Objective key = '<final beam file>:<param>'. Adjust the beam-file prefix to the last
    # screen/marker actually dumped in your run (printed by the sanity check below).
    objectives={"JFEL-FEL-SIM-MARK-01:enx": "MINIMIZE"},
)

> **Tip.** Before launching the optimiser, run the evaluator once by hand to see exactly which keys
> it returns — the objective key above must be one of them. This one call is a full track, so it is
> also your smoke test that the `.def`, lattice and SimCodes are all wired up.

In [ ]:
sample = evaluate({
    "JFEL-S02-MAG-QUAD-01:k1l": 0.0,
    "JFEL-S02-MAG-QUAD-02:k1l": 0.0,
})
print("available objective keys:")
for k in sorted(sample):
    print("   ", k, "=", sample[k])

In [ ]:
X = Xopt(
    vocs=vocs,
    evaluator=Evaluator(function=evaluate),
    generator=ExpectedImprovementGenerator(vocs=vocs),
)

# A handful of random seeds, then let Bayesian optimisation take over.
X.random_evaluate(3)
for _ in range(15):
    X.step()

print(X.data.sort_values("JFEL-FEL-SIM-MARK-01:enx").head())

## Part 2 — `nelder_mead` for a self-contained objective

For small problems, or when you would rather not pull in Xopt, `simba.Modules.optimisation.nelder_mead`
is a plain simplex optimiser: give it a function that maps a numpy array of variables to a scalar
score, and a starting point. Here we write the objective ourselves — track the machine for a given
pair of quad strengths and return the emittance to minimise.

In [ ]:
import simba.Framework as fw
from simba.Framework import load_directory
from simba.Modules.optimisation.nelder_mead import nelder_mead

framework = fw.Framework(
    simcodes=SIMCODES,
    directory="./opt_nm",
    master_lattice=LATTICE,
    generator_defaults="jfel.yaml",
    clean=True,
    verbose=False,
)
framework.loadSettings("Lattices/jfel_combined.def")
framework.change_generator("ASTRA")
framework.generator.load_defaults("jfel_400_3ps")
framework.generator.number_of_particles = 2 ** (3 * 2)

In [ ]:
QUADS = ["JFEL-S02-MAG-QUAD-01", "JFEL-S02-MAG-QUAD-02"]

def objective(x):
    """Track the injector+linac for quad strengths x, return final normalised emittance."""
    for name, k1l in zip(QUADS, x):
        framework.modifyElement(name, "k1l", float(k1l))
    framework.track(startfile="generator", endfile="Linac")
    fwdir = load_directory("./opt_nm", beams=True)
    final = fwdir.beams[-1]
    return abs(float(final.enx))

best_x, best_score = nelder_mead(
    objective,
    np.array([0.0, 0.0]),
    step=0.2,
    max_iter=25,
)
print("best quad strengths:", best_x, "-> enx =", best_score)

> **Cost warning.** Every objective evaluation is a *full* tracking run. Keep the particle count and
> `max_iter` low while you get the objective and bounds right, then scale up. Choosing a fast code for
> the section under optimisation (e.g. Ocelot/Elegant rather than a space-charge ASTRA run) makes the
> loop far more practical — see `settings["code"]` in `xopt_optimisation`, or
> `change_Lattice_Code` in the [chaining example](start_to_end_chain.ipynb).

### Cleanup

In [ ]:
import shutil
for d in ("./opt_run", "./opt_nm"):
    shutil.rmtree(d, ignore_errors=True)

### Recap

* An optimisation in `SIMBA` is *tracking wrapped as an objective function*.
* `xopt_optimisation` gives you that wrapper ready-made for Xopt: declare variables and objectives in
  a `VOCS` using the `element:param` / `beam:param` key conventions.
* `nelder_mead` is the lightweight alternative when you want to write the objective by hand.
* Runs are expensive — start small, pick a fast code for the loop, and scale up once it converges.